# 05 — Transformers (mBERT, XLM-R-base, XLM-R-large)

Order: mBERT → XLM-R-base → XLM-R-large. Each takes ~10–30 min per language on a T4 / A100.

If you run out of GPU memory on `xlm-roberta-large`, drop `batch_size` to 8 and set `gradient_accumulation_steps=2` in `C.TransformerConfig`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src import config as C
from src.data_utils import get_split
from src.evaluate import evaluate_and_log
from src.models import transformer

In [ ]:
MODELS = {
    'mbert': 'bert-base-multilingual-cased',
    'xlmr-base': 'xlm-roberta-base',
    'xlmr-large': 'xlm-roberta-large',  # comment out if GPU is small
}

for short_name, hf_name in MODELS.items():
    for lang in C.LANGUAGES:
        print(f'\n=== {short_name} :: {lang.upper()} ===')
        try:
            tr, va, te = get_split(lang)
        except FileNotFoundError as e:
            print(f'  skipped: {e}'); continue
        cfg = C.TransformerConfig(model_name=hf_name)
        bundle = transformer.train(tr, va, lang, cfg=cfg)
        y_pred = transformer.predict(bundle, te, lang)
        metrics = evaluate_and_log(te['label'].values, y_pred, model_name=short_name, lang=lang)
        print('  metrics:', {k: round(v, 4) for k, v in metrics.items()})
        transformer.save(bundle, C.RESULTS_DIR / 'checkpoints' / f'{short_name}_{lang}')